# AI Based Exam Anxiety Detector - Model Training

This notebook covers MILESTONE 2, 3, and 4:
- Dataset Loading and EDA
- Data Preprocessing & Label Mapping
- BERT Model Selection & Training
- Model Evaluation and Saving

In [ ]:
!pip install transformers torch pandas scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments
from torch.utils.data import Dataset

## MILESTONE 2 - Dataset Collection and Exploratory Data Analysis

In [ ]:
# Activity 2.2: Dataset Loading
# Ensure 'sample_dataset.csv' is uploaded to Colab
df = pd.read_csv('sample_dataset.csv')
print("Activity 2.3: Understanding Dataset Structure")
print(df.head())

print("\nActivity 2.4: Checking Missing Values")
print(df.isnull().sum())

print("\nActivity 2.5: Class Distribution Analysis")
print(df['label'].value_counts())

## MILESTONE 3 - Data Preprocessing & Label Mapping

In [ ]:
# Activity 3.1: Handling Missing Text Data
df = df.dropna()

# Activity 3.3 & 3.4: Anxiety Level Label Mapping & Creating Numerical Labels
label_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
df['numerical_label'] = df['label'].map(label_mapping)

print("Activity 3.5: Validation of Label Mapping")
print(df[['label', 'numerical_label']].drop_duplicates())

# Activity 3.6: Final Dataset Preparation
X = df['text'].tolist()
y = df['numerical_label'].tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## MILESTONE 4 - BERT Model Selection & Training

In [ ]:
# Activity 4.1 & 4.4: Selection of BERT Model and Tokenization
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)

train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(X_test, truncation=True, padding=True, max_length=128)

class AnxietyDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = AnxietyDataset(train_encodings, y_train)
test_dataset = AnxietyDataset(test_encodings, y_test)

# Activity 4.5: Model Training Process
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=3)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=10,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    evaluation_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

## MILESTONE 5 - Model Evaluation and Saving

In [ ]:
# Activity 4.6 & 5.1: Model Evaluation and Saving the Trained Model
eval_results = trainer.evaluate()
print(f"Evaluation Results: {eval_results}")

model.save_pretrained('./saved_model')
tokenizer.save_pretrained('./saved_model')
print("Model saved to ./saved_model directory. You can download this directory for the backend.")